# LOAD DATA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif"], # Matches Times New Roman
    "font.size": 10,                    # Standard for most papers
    "axes.labelsize": 10,
    "xtick.labelsize": 8,              
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "text.usetex": False,               
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})
df_deep_train=pd.read_csv("/kaggle/input/multiclass-deep-features/train_features_multiclass.csv")
df_deep_test=pd.read_csv("/kaggle/input/multiclass-deep-features/test_features_multiclass.csv")
df_handcrafted_train=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Hnadcrafted_Train_features_multiclass.csv")
df_handcrafted_test=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Handcraftred_Test_features_multiclass.csv")

print("--- Deep Features Labels ---")
print(df_deep_train['label'].value_counts().sort_index())

print("\n--- Handcrafted Features Labels ---")
print(df_handcrafted_train['label'].value_counts().sort_index())

label_fix_map = {0: 3, 1: 2, 2: 1, 3: 0} 

df_deep_train['label'] = df_deep_train['label'].map(label_fix_map)
df_deep_test['label']  = df_deep_test['label'].map(label_fix_map)

print("✅ Labels remapped. Re-checking consistency...")

X_deep= pd.concat((df_deep_train,df_deep_test),axis=0)
X_deep_final = X_deep.drop(columns=['image_id','label'])
y_deep = X_deep['label'].to_numpy().ravel()

X_handcrafted= pd.concat((df_handcrafted_train,df_handcrafted_test),axis=0)
X_handcrafted_final = X_handcrafted.drop(columns=['id','label'])
y_handcrafted = X_handcrafted['label'].to_numpy().ravel()

df_deep_test['id']=df_deep_test['image_id']
df_deep_train['id']=df_deep_train['image_id']
   
df_fusion_train=pd.merge(df_deep_train,df_handcrafted_train,on='id',how='inner')
df_fusion_test=pd.merge(df_deep_test,df_handcrafted_test,on='id',how='inner')

X_fusion_train=df_fusion_train.drop(columns=['image_id','id','label_y','label_x'])
X_fusion_test=df_fusion_test.drop(columns=['image_id','id','label_y','label_x'])

y_train=df_fusion_train['label_y']
y_test=df_fusion_test['label_y']

X_total = np.concatenate((X_fusion_train, X_fusion_test), axis=0)
y_total = np.concatenate((y_train, y_test), axis=0)

In [ ]:
# Check for Label Consistency
labels_match_train = (df_fusion_train['label_x'] == df_fusion_train['label_y']).all()
labels_match_test = (df_fusion_test['label_x'] == df_fusion_test['label_y']).all()

if not labels_match_train or not labels_match_test:
    print("CRITICAL WARNING: Label Mismatch Detected!")
else:
    print("Labels are consistent.")

# SHAP VISUALISATION

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Generating Multiclass SHAP Visualizations...")


class_names = ["Pituitary", "No Tumor", "Meningioma", "Glioma"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_fusion_train)
X_test_scaled  = scaler.transform(X_fusion_test)

feature_names = list(X_fusion_train.columns)


lr_raw = LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1)
lr_raw.fit(X_train_scaled, y_train)


explainer = shap.LinearExplainer(lr_raw, X_train_scaled, feature_dependence="independent")
shap_values = explainer.shap_values(X_test_scaled)

# Convert to DataFrame with feature names for clear plotting
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Bar Plot
plt.figure()
shap.summary_plot(
    shap_values, 
    X_test_df, 
    plot_type="bar", 
    class_names=class_names,
    max_display=20,
    show=True
)

# Dot Plot
plt.figure()
shap.summary_plot(
    shap_values[3], 
    X_test_df, 
    max_display=20,
    show=True
)


is_deep = np.array(["deep" in col.lower() or "resnet" in col.lower() or i < 2048 for i, col in enumerate(feature_names)])
is_radiomic = ~is_deep

# Mean absolute SHAP values across all test samples and classes
if isinstance(shap_values, list): # List of arrays (one per class)
    total_abs_shap = np.sum([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    total_abs_shap = np.abs(shap_values).mean(axis=0)

deep_share = total_abs_shap[is_deep].sum() / total_abs_shap.sum() * 100.0
rad_share  = total_abs_shap[is_radiomic].sum() / total_abs_shap.sum() * 100.0

print("\nMODALITY ATTRIBUTION SHARE")
print(f"Deep Feature Share       : {deep_share:.2f}%")
print(f"Handcrafted Radiomic Share: {rad_share:.2f}%")